<a href="https://colab.research.google.com/github/nikolakr7/learningMLsys/blob/main/01_vectoradd_relu_Cbasics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## GPUMODE Lecture 3 mini-project: vector add and ReLU

Hand-written CUDA kernels, benchmarked against PyTorch on a Colab T4.
Each time is the average of 100 runs after 10 warm-up runs.

### Results (10M floats)

| Kernel | Mine | PyTorch | CPU, 1 core (-O2) |
| :-- | :-- | :-- | :-- |
| Vector add | 0.457 ms | 0.493 ms | 28.04 ms |
| ReLU | 0.318 ms | 0.339 ms | not measured |

Both kernels are limited by memory speed, not math. Vector add moves
120 MB in 0.457 ms, about 260 GB/s, close to the T4's 320 GB/s limit.

### Why are my kernels slightly faster than PyTorch's? (testing on vector add)

| n | Mine | PyTorch | Gap |
| :-- | :-- | :-- | :-- |
| 1 million | 0.050 ms | 0.058 ms | 0.008 ms |
| 10 million | 0.457 ms | 0.493 ms | 0.036 ms |
| 50 million | 2.277 ms | 2.450 ms | 0.173 ms |

The gap grows with n, so it comes from the kernel itself, not per-call
overhead. From the first two rows I predicted a gap of about 0.16 ms at
50 million and measured 0.173 ms.

# First I have to learn C basics

In [ ]:
%%writefile hello.c
#include <stdio.h>

int main() {
    printf("Hello from C\n");
    return 0;
}

Writing hello.c


In [ ]:
!gcc hello.c -o hello && ./hello

Hello from C


In [ ]:
%%writefile lesson1.c
#include <stdio.h>

float relu(float x) {
  return x > 0 ? x : 0;
}

int blocks_needed(int n, int threads) {
  return (n + threads - 1) / threads;
}

int main() {
   printf("%d\n", 7 / 2);
   printf("%f\n", 7 / 2.0f);
   int total = 0;
   printf("%f\n", relu(-2.5f));
   printf("%f\n", relu(3.0f));
   printf("%d\n", blocks_needed(1000,256));
   printf("%d\n", blocks_needed(1024,256));
   printf("%d\n", blocks_needed(1025,256));

   for (int i = 0; i < 100; i++) {
      total += i;
   }

   printf("%d\n", total);
}

Overwriting lesson1.c


In [ ]:
!gcc lesson1.c -o lesson1 && ./lesson1

3
3.500000
0.000000
3.000000
4
4
5
4950


In [ ]:
%%writefile pointers.c
#include <stdio.h>

int main() {
    float x = 2.5f;
    float *p = &x;            // p holds the address of x

    printf("x is %f\n", x);
    printf("address of x is %p\n", p);
    printf("value at that address is %f\n", *p);

    *p = 10.0f;               // go to that address and change what's there
    printf("x is now %f\n", x);
    return 0;
}

Overwriting pointers.c


In [ ]:
!gcc pointers.c -o pointers && ./pointers

x is 2.500000
address of x is 0x7ffe9aece1ec
value at that address is 2.500000
x is now 10.000000


In [ ]:
%%writefile pointers.c
#include <stdio.h>

void double_it(float *x) {
  *x *= 2.0f;
}

void scale(float *arr, int n, float factor) {
  for (int i=0; i<n; i++) {
    arr[i] *= factor;
  }
}

void add_arrays(float *a, float *b, float *out, int n) {
  for (int i=0; i<n; i++) {
    out[i] = a[i] + b[i];
  }
}

void print_array(float *arr, int n) {
    for (int i = 0; i < n; i++) {
        printf("%f ", arr[i]);
    }
    printf("\n");
}

int main() {
   float x = 1.0f;
   float *p = &x;
   *p = *p + 4.0f;
   printf("%f\n", x);
   float y = 5.0f;
   printf("%f\n", y);
   double_it(&y);
   printf("%f\n", y);
   float arr[] = {1.0f, 2.0f, 3.0f, 4.0f, 5.0f};
   scale(arr, 5, 2.0f);
   print_array(arr,5);
   printf("%f\n", arr[10]); // going out of bounds on purpose to see the output
   float a[] = {1.0f, 2.0f, 3.0f, 4.0f, 5.0f};
   float b[] = {4.0f, 5.0f, 6.0f, 7.0f, 8.0f};
   float out[5];
   add_arrays(a,b,out,5);
   print_array(out,5);
   return 0;

}

Overwriting pointers.c


In [ ]:
!gcc pointers.c -o pointers && ./pointers

5.000000
5.000000
10.000000
2.000000 4.000000 6.000000 8.000000 10.000000 
0.000000
5.000000 7.000000 9.000000 11.000000 13.000000 


In [ ]:
%%writefile malloc_demo.c
#include <stdio.h>
#include <stdlib.h>

int main() {
    int n = 10;
    float *a = (float *)malloc(n * sizeof(float));

    for (int i = 0; i < n; i++) {
        a[i] = i * 1.5f;
    }
    for (int i = 0; i < n; i++) {
        printf("%.1f ", a[i]);
    }
    printf("\n");

    free(a);
    return 0;
}

Writing malloc_demo.c


In [ ]:
!gcc malloc_demo.c -o malloc_demo && ./malloc_demo

0.0 1.5 3.0 4.5 6.0 7.5 9.0 10.5 12.0 13.5 


In [ ]:
%%writefile array_test.c
#include <stdio.h>

int main() {
   float big[100000000];
   big[0] = 1.0f;
   printf("%f\n", big[0]);
}

Overwriting array_test.c


In [ ]:
!gcc array_test.c -o array_test && ./array_test

# Now getting into vectoradd and relu stuff, starting by doing it on the cpu to compare to gpu later

In [10]:
%%writefile vector_add_cpu.c
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

void add_arrays(float *a, float *b, float *out, int n) {
  for (int i=0; i<n; i++) {
    out[i] = a[i] + b[i];
  }
}

int main() {
  int n = 10000000;
  float *a = (float *)malloc(n * sizeof(float));
  float *b = (float *)malloc(n * sizeof(float));
  float *out = (float *)malloc(n * sizeof(float));
  for (int i=0; i<n; i++) {
    a[i] = 1.0f;
    b[i] = 2.0f;
  }
  clock_t start = clock();
  add_arrays(a,b,out,n);
  clock_t end = clock();
  double ms = 1000.0 * (end - start) / CLOCKS_PER_SEC;
  printf("took %.2f ms\n", ms);
  int not_three = 0;
  for (int i=0; i<n; i++) {
    if (out[i] != 3.0f) {
      not_three += 1;
    }
  }
  printf("%d\n", not_three);
  free(a);
  free(b);
  free(out);
  }


Overwriting vector_add_cpu.c


In [11]:
!gcc -O2 vector_add_cpu.c -o vector_add_cpu && ./vector_add_cpu

took 55.72 ms
0


In [2]:
%%writefile scale.cu
#include <stdio.h>
#include <stdlib.h>

__global__ void scale_kernel(float *arr, int n, float factor) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        arr[i] = arr[i] * factor;
    }
}

int main() {
    int n = 10000000;
    size_t bytes = n * sizeof(float);

    // 1. Make the data on the CPU
    float *h_arr = (float *)malloc(bytes);
    for (int i = 0; i < n; i++) {
        h_arr[i] = 1.0f;
    }

    // 2. Reserve memory on the GPU
    float *d_arr;
    cudaMalloc((void **)&d_arr, bytes);

    // 3. Copy the data from CPU to GPU
    cudaMemcpy(d_arr, h_arr, bytes, cudaMemcpyHostToDevice);

    // 4. Launch the kernel
    int threads = 256;
    int blocks = (n + threads - 1) / threads;
    scale_kernel<<<blocks, threads>>>(d_arr, n, 3.0f);

    // 5. Wait for the GPU to finish
    cudaDeviceSynchronize();

    // 6. Copy the result from GPU back to CPU
    cudaMemcpy(h_arr, d_arr, bytes, cudaMemcpyDeviceToHost);

    // 7. Check the result
    int wrong = 0;
    for (int i = 0; i < n; i++) {
        if (h_arr[i] != 3.0f) wrong++;
    }
    printf("wrong: %d\n", wrong);

    // 8. Clean up
    cudaFree(d_arr);
    free(h_arr);
    return 0;
}

Writing scale.cu


In [3]:
!nvcc scale.cu -o scale && ./scale

wrong: 0


# writing a vector add on gpu then comparing to pytorch and cpu

In [55]:
%%writefile vector_add.cu
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

double now_ms() {
    struct timespec t;
    clock_gettime(CLOCK_MONOTONIC, &t);
    return t.tv_sec * 1000.0 + t.tv_nsec / 1000000.0;
}

__global__ void vector_add_kernel(float *a, float *b, float *out, int n) {
  int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < n) {
    out[i] = a[i] + b[i];
  }
}

int main() {
  int n = 10000000;
  size_t bytes = n * sizeof(float);
  // Allocate memory for each array on the CPU
  float *h_a = (float *)malloc(bytes);
  float *h_b = (float *)malloc(bytes);
  float *h_out = (float *)malloc(bytes);

  for (int i = 0; i < n; i++) {
        h_a[i] = i;
        h_b[i] = 2.0f * i;
  }
  // Allocate memory for each array on the GPU
  float *d_a;
  cudaMalloc((void **)&d_a, bytes);
  float *d_b;
  cudaMalloc((void **)&d_b, bytes);
  float *d_out;
  cudaMalloc((void **)&d_out, bytes);
  // Copy data from CPU to GPU
  cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
  cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);
  // Launching the kernel
  int threads = 256;
  int blocks = (n + threads - 1) / threads;
  // warm-up runs, not timed
  for (int r = 0; r < 10; r++) {
      vector_add_kernel<<<blocks, threads>>>(d_a, d_b, d_out, n);
  }
  cudaDeviceSynchronize();
  // timed runs
  int runs = 100;
  double start = now_ms();
  for (int r = 0; r < runs; r++) {
      vector_add_kernel<<<blocks, threads>>>(d_a, d_b, d_out, n);
  }
  cudaDeviceSynchronize();
  double end = now_ms();
  printf("avg per run: %.3f ms\n", (end - start) / runs);
  // Copy the output array from GPU back to CPU
  cudaMemcpy(h_out, d_out, bytes, cudaMemcpyDeviceToHost);
  // Check the result
  int wrong = 0;
    for (int i = 0; i < n; i++) {
        if (h_out[i] != h_a[i] + h_b[i]) wrong++;
    }
    printf("wrong: %d\n", wrong);
  // Dont forget to clean up
  cudaFree(d_a);
  cudaFree(d_b);
  cudaFree(d_out);
  free(h_a);
  free(h_b);
  free(h_out);
  return 0;
}



Overwriting vector_add.cu


In [56]:
!nvcc vector_add.cu -o vector_add && ./vector_add

avg per run: 0.458 ms
wrong: 0


# writing relu on gpu then comparing to pytorch and cpu

In [39]:
%%writefile relu.cu
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

double now_ms() {
    struct timespec t;
    clock_gettime(CLOCK_MONOTONIC, &t);
    return t.tv_sec * 1000.0 + t.tv_nsec / 1000000.0;
}

__global__ void relu_kernel(float *in, float *out, int n) {
  int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < n) {
    out[i] = in[i] > 0 ? in[i] : 0;
  }
}

int main() {
  int n = 10000000;
  size_t bytes = n * sizeof(float);
  // Allocate memory for each array on the CPU
  float *h_in = (float *)malloc(bytes);
  float *h_out = (float *)malloc(bytes);
  // Fill half the input array with negatives and half with positives
  for (int i = 0; i < n; i++) {
        h_in[i] = i - 5000000;
  }
  // Allocate memory for each array on the GPU
  float *d_in;
  cudaMalloc((void **)&d_in, bytes);
  float *d_out;
  cudaMalloc((void **)&d_out, bytes);
  // Copy data from CPU to GPU
  cudaMemcpy(d_in, h_in, bytes, cudaMemcpyHostToDevice);
  // Launching the kernel
  int threads = 256;
  int blocks = (n + threads - 1) / threads;
  // warm-up runs, not timed
  for (int r = 0; r < 10; r++) {
      relu_kernel<<<blocks, threads>>>(d_in, d_out, n);
  }
  cudaDeviceSynchronize();
  // timed runs
  int runs = 100;
  double start = now_ms();
  for (int r = 0; r < runs; r++) {
      relu_kernel<<<blocks, threads>>>(d_in, d_out, n);
  }
  cudaDeviceSynchronize();
  double end = now_ms();
  printf("avg per run: %.3f ms\n", (end - start) / runs);
  // Copy data back to CPU
  cudaMemcpy(h_out, d_out, bytes, cudaMemcpyDeviceToHost);
  // Check each output against what the CPU says ReLU of that input should be
  int wrong = 0;
  for (int i = 0; i < n; i++) {
    if (h_out[i] != (h_in[i] > 0 ? h_in[i] : 0)) wrong++;
  }
  printf("wrong: %d\n", wrong);
  // Dont forget to clean up
  free(h_in);
  free(h_out);
  cudaFree(d_in);
  cudaFree(d_out);
  return 0;
}




Overwriting relu.cu


In [40]:
!nvcc relu.cu -o relu && ./relu

avg per run: 0.317 ms
wrong: 0


# pytorch vector add

In [57]:
import torch, time

n = 10_000_000
a = torch.ones(n, device="cuda")
b = torch.full((n,), 2.0, device="cuda")
out = torch.empty(n, device="cuda")

# warm-up runs, not timed
for _ in range(10):
    torch.add(a, b, out=out)
torch.cuda.synchronize()

# timed runs
runs = 100
start = time.perf_counter()
for _ in range(runs):
    torch.add(a, b, out=out)
torch.cuda.synchronize()
end = time.perf_counter()
print(f"avg per run: {(end - start) / runs * 1000:.3f} ms")

avg per run: 0.493 ms


# pytorch relu

In [38]:
import torch, time

n = 10_000_000
in_ = torch.randn(n, device="cuda")
out = torch.empty(n, device="cuda")

# warm-up runs, not timed
for _ in range(10):
    torch.clamp(in_, min=0, out=out)
torch.cuda.synchronize()

# timed runs
runs = 100
start = time.perf_counter()
for _ in range(runs):
    torch.clamp(in_, min=0, out=out)
torch.cuda.synchronize()
end = time.perf_counter()
print(f"avg per run: {(end - start) / runs * 1000:.3f} ms")

avg per run: 0.338 ms
